# Circuit Detective Phase 1 Pilot

This notebook is the first training scaffold for the L1 induction-localization pilot: `Qwen/Qwen3.5-2B` + TRL `GRPOTrainer` + 4-bit LoRA on the Phase 1 Circuit Detective environment.

It assumes you are running from a clone of this repo. The environment itself stays OpenEnv-valid, but the trainer uses a direct Python wrapper with explicit public tool methods because that matches TRL's `environment_factory` contract and removes an unnecessary websocket failure surface for the first pilot.

## Install

The main notebook runtime keeps `openenv-core` and trainer dependencies. TransformerLens is installed into a sidecar `.venv-tlens` because its current dependency constraints conflict with the OpenEnv stack.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

print("repo root:", Path.cwd())

# Run in a fresh notebook runtime.
%pip install -q "openenv-core[core]==0.2.3" "trl==1.1.0" "datasets>=3.0.0" "matplotlib>=3.8" "jmespath" "peft" "bitsandbytes" "uv"
%pip install -q -e .
!uv venv .venv-tlens --python 3.11
!HF_HUB_DISABLE_XET=1 uv pip install -q --torch-backend cpu --python .venv-tlens/bin/python "transformer-lens==2.18.0"

In [ ]:
import math

import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

from circuit_detective.phase1_grpo import (
    CircuitDetectiveToolEnv,
    build_phase1_dataset,
    consume_reward_trace,
    reward_func,
    reset_reward_trace,
)

model_name = "Qwen/Qwen3.5-2B"
max_seq_length = 1024
max_steps = 100
lora_rank = 8
repeats_per_prompt = 16  # Smoke-test default. Raise to 64 for a longer pilot.
num_generations = 4
eval_generations = num_generations  # Required for TRL environment_factory eval path.
eval_prompts = 4
output_dir = "outputs/phase1_qwen35_2b_grpo"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_rank,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
train_dataset = build_phase1_dataset(repeats_per_prompt=repeats_per_prompt)
eval_count = max(eval_prompts, eval_generations)
eval_count = math.ceil(eval_count / eval_generations) * eval_generations
base_eval_dataset = build_phase1_dataset(repeats_per_prompt=1)
eval_repeats = math.ceil(eval_count / len(base_eval_dataset))
eval_dataset = build_phase1_dataset(repeats_per_prompt=eval_repeats)
eval_dataset = eval_dataset.select(range(eval_count))
print(f"train_rows={len(train_dataset)} eval_rows={len(eval_dataset)}")
train_dataset[0]

In [ ]:
# Sanity-check one live environment episode before launching training.
import json

debug_env = CircuitDetectiveToolEnv()
print(debug_env.reset())
scores_payload = json.loads(debug_env.inspect_induction_scores(top_k=3))
print(scores_payload)
top_head = scores_payload["result"]["scores"][0]["head_id"]
print(debug_env.submit_circuit(heads=[top_head]))
debug_env._close()

In [ ]:
training_args = GRPOConfig(
    output_dir=output_dir,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=1,
    bf16=bf16,
    fp16=not bf16,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=eval_generations,
    gradient_accumulation_steps=1,
    num_generations=num_generations,
    num_generations_eval=eval_generations,
    generation_batch_size=num_generations,
    max_completion_length=384,
    max_steps=max_steps,
    save_steps=max(max_steps // 2, 1),
    max_grad_norm=0.1,
    report_to="none",
    log_completions=True,
    chat_template_kwargs={"enable_thinking": False},
    use_vllm=False,
    max_tool_calling_iterations=4,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_func],
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    environment_factory=CircuitDetectiveToolEnv,
    peft_config=peft_config,
)

In [ ]:
def summarize_reward_trace(records, prefix):
    if not records:
        return {
            f"{prefix}_rollouts": 0.0,
            f"{prefix}_mean_reward": 0.0,
            f"{prefix}_submit_rate": 0.0,
            f"{prefix}_success_rate": 0.0,
            f"{prefix}_mean_f1": 0.0,
            f"{prefix}_mean_terminal_reward": 0.0,
            f"{prefix}_mean_tool_calls": 0.0,
            f"{prefix}_probe_rate": 0.0,
            f"{prefix}_inspect_rate": 0.0,
            f"{prefix}_ablate_rate": 0.0,
        }
    count = float(len(records))
    mean = lambda key: sum(float(record.get(key, 0.0)) for record in records) / count
    rate = lambda key: sum(1.0 for record in records if bool(record.get(key))) / count
    return {
        f"{prefix}_rollouts": count,
        f"{prefix}_mean_reward": mean("reward"),
        f"{prefix}_submit_rate": rate("submitted"),
        f"{prefix}_success_rate": rate("correct"),
        f"{prefix}_mean_f1": mean("f1"),
        f"{prefix}_mean_terminal_reward": mean("terminal_reward"),
        f"{prefix}_mean_tool_calls": mean("tool_calls"),
        f"{prefix}_probe_rate": rate("used_probe"),
        f"{prefix}_inspect_rate": rate("used_inspect"),
        f"{prefix}_ablate_rate": rate("used_ablate"),
    }

def evaluate_with_rollout_metrics(trainer, metric_key_prefix):
    reset_reward_trace()
    metrics = trainer.evaluate(metric_key_prefix=metric_key_prefix)
    rollout_records = consume_reward_trace()
    metrics.update(summarize_reward_trace(rollout_records, metric_key_prefix))
    metrics[f"{metric_key_prefix}_sample_rollouts"] = rollout_records[:8]
    return dict(metrics)

before_metrics = evaluate_with_rollout_metrics(trainer, "eval_before")
reset_reward_trace()
trainer.train()
after_metrics = evaluate_with_rollout_metrics(trainer, "eval_after")
trainer.save_model(f"{output_dir}/final_adapter")
print("before:", before_metrics)
print("after:", after_metrics)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

artifacts_dir = Path("artifacts/phase1")
artifacts_dir.mkdir(parents=True, exist_ok=True)
log_history = trainer.state.log_history
available_keys = sorted({key for entry in log_history for key in entry})
print("available log keys:", available_keys)

def save_curve(key: str, path: Path, title: str) -> None:
    xs = [entry["step"] for entry in log_history if "step" in entry and key in entry]
    ys = [entry[key] for entry in log_history if "step" in entry and key in entry]
    if not xs:
        raise ValueError(f"No values found for {key!r}. Available keys: {available_keys}")
    plt.figure(figsize=(6, 4))
    plt.plot(xs, ys)
    plt.xlabel("step")
    plt.ylabel(key)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

save_curve("loss", artifacts_dir / "phase1_loss_curve.png", "Phase 1 Loss")

reward_key = next(
    (
        key
        for key in [
            "reward",
            "rewards",
            "mean_reward",
            "objective/reward",
            "reward/mean",
        ]
        if any(key in entry for entry in log_history)
    ),
    None,
)

if reward_key is None:
    print("No reward-like key found automatically. Inspect available_keys above and rerun save_curve with the correct key.")
else:
    save_curve(
        reward_key,
        artifacts_dir / "phase1_reward_curve.png",
        f"Phase 1 Reward ({reward_key})",
    )
    print("saved:", artifacts_dir / "phase1_reward_curve.png")

(artifacts_dir / "phase1_eval_metrics.json").write_text(
    json.dumps({"before": before_metrics, "after": after_metrics}, indent=2, sort_keys=True),
    encoding="utf-8",
)
print("saved:", artifacts_dir / "phase1_eval_metrics.json")
print("saved:", artifacts_dir / "phase1_loss_curve.png")